In [1]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface, Probe
from scipy.io import loadmat
import pickle
import spikeinterface.exporters as sexp

from scipy.io import loadmat
from utils_clique import (
    neuron_inf_dict_to_dataframe,
    get_recording_clique,
    plot_cliques,
    prepare_training_data,
    train_autosort_model,
    build_sliding_cliques
)
probe_data = loadmat("/media/ubuntu/sda/duan/raw_data/chanMap_DCX_5mm.mat")
probe_x = probe_data['xcoords']
probe_y = probe_data['ycoords']

probe_position = pd.DataFrame(probe_x)
probe_position[1] = probe_y
probe_position['chan_map'] = probe_data['chanMap0ind'].astype(int)

chan_map = pd.read_csv('/media/ubuntu/sda/duan/raw_data/ch_map_R.csv')
merged = chan_map.merge(probe_position, left_on='probeloc', right_on='chan_map')\
                 .iloc[chan_map.index]\
                 .reset_index(drop=True)

probe = Probe()
probe.set_contacts(positions=merged.iloc[:, 2:4])
probe.set_device_channel_indices(range(256))

/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
recording_raw = se.read_intan(f"/home/ubuntu/Documents/jct/project/251205/M190011_260121_150111_merged_130.rhd", stream_id= '0', ignore_integrity_checks=True)

print('read success')

recording_raw = spre.unsigned_to_signed(recording_raw)
recording_raw = spre.resample(recording_raw, 10000)

recording_recorded = spre.bandpass_filter(recording_raw, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_recorded, freq=50)
recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

recording_f = recording_f.set_probegroup(probe)

rec_params_raw = pd.read_csv("/media/ubuntu/sda/duan/result/260121/rec_params.csv")
rec_params_raw = rec_params_raw[rec_params_raw['bhv_codes'] == 10]

original_fs = 30000
target_fs = 10000
fs_ratio = original_fs / target_fs
rec_params_raw['rec_codes_points_10000'] = (rec_params_raw['rec_codes_points'] / fs_ratio).astype(int)
rec_params_raw = rec_params_raw[(rec_params_raw['trial_ids'] >= 300) & (rec_params_raw['trial_ids'] < 5300)]
start_sample = rec_params_raw['rec_codes_points_10000'].iloc[0]
end_sample = rec_params_raw['rec_codes_points_10000'].iloc[-1]

recording_segment = recording_f.frame_slice(start_frame=start_sample, end_frame=end_sample)


read success


In [3]:
output_folder = '/media/ubuntu/sda/visual_generation/results/neuroscroll_260122'
probe.set_contact_ids(recording_segment.channel_ids)

cliques = build_sliding_cliques(
    probe,
    clique_size=32,
    min_size=25,
    min_overlap=6,
    target_groups=10,
)

clique_info = {
    'cliques': cliques,  # List of CliqueInfo objects
    'clique_params': {
        'clique_size': 32,
        'min_size': 25,
        'min_overlap': 6,
        'target_groups': 10,
    },
    'probe_df': probe.to_dataframe(),  # Probe dataframe for verification
}

clique_info_path = f'{output_folder}/clique_info.pkl'
with open(clique_info_path, 'wb') as f:
    pickle.dump(clique_info, f)

plot_cliques(probe, cliques, f'{output_folder}/cliques.pdf')

[INFO] Built 10 cliques (target 10)
       Clique 00: channels 204-39 (32 channels)
       Clique 01: channels 74-81 (32 channels)
       Clique 02: channels 174-227 (32 channels)
       Clique 03: channels 33-88 (32 channels)
       Clique 04: channels 69-253 (32 channels)
       Clique 05: channels 229-232 (32 channels)
       Clique 06: channels 35-24 (32 channels)
       Clique 07: channels 207-141 (32 channels)
       Clique 08: channels 134-109 (32 channels)
       Clique 09: channels 148-115 (32 channels)

Clique可视化PDF已保存至: /media/ubuntu/sda/visual_generation/results/neuroscroll_260122/cliques.pdf


In [6]:
for clique in cliques:
    clique_id = clique.clique_id

    recording_clique = get_recording_clique(recording_segment, clique)
    output_folder = f'/media/ubuntu/sda/visual_generation/results/neuroscroll_260122/mountainsort/clique_{clique_id}'
    os.makedirs(output_folder, exist_ok=True)
    recording_preprocessed = recording_clique.save(format="binary", n_jobs = 30)   

    default_params = {
            'detect_sign': 0,  
            'adjacency_radius': 120, 
            'freq_min': 300,  
            'freq_max': 3000,
            'filter': True,
            'whiten': True,  
            'num_workers': 30,
            'clip_size': 50,
            'detect_threshold': 5,
            'detect_interval': 3,  
        }
    sorting_mountainsort = ss.run_sorter(sorter_name='mountainsort4',
                                    recording=recording_preprocessed,
                                    remove_existing_folder='True',
                                    folder=output_folder,
                                    **default_params)


    analyzer_mountainsort = si.create_sorting_analyzer(
        sorting=sorting_mountainsort, 
        recording=recording_preprocessed, 
        format='binary_folder', 
        folder=output_folder + '/analyzer_kilosort4_binary'
    )

    extensions_to_compute = [
        "random_spikes",
        "waveforms",
        "noise_levels",
        "templates",
        "unit_locations",
        "spike_locations",
        "correlograms",
        "template_similarity"
    ]

    extension_params = {
        "unit_locations": {"method": "center_of_mass"},
        "spike_locations": {"ms_before": 0.1},
        "correlograms": {"bin_ms": 0.1},
        "template_similarity": {"method": "cosine_similarity"}
    }

    analyzer_mountainsort.compute(extensions_to_compute, extension_params=extension_params, n_jobs = 30)

    spikes_path = output_folder + "/analyzer_kilosort4_binary/sorting/spikes.npy"
    spikes = np.load(spikes_path)

    total_samples = recording_f.get_num_samples()

    first_spike_valid = spikes[0]['sample_index'] >= 0
    last_spike_valid = spikes[-1]['sample_index'] < total_samples

    if not first_spike_valid or not last_spike_valid:
        valid_mask = (spikes['sample_index'] >= 0) & (spikes['sample_index'] < total_samples)
        spikes_filtered = spikes[valid_mask]
        
        # 保存过滤后的spikes
        np.save(spikes_path, spikes_filtered)
        print(f"删除了 {len(spikes) - len(spikes_filtered)} 个无效的spike")
        print(f"原始spike数量: {len(spikes)}, 过滤后: {len(spikes_filtered)}")
    else:
        print("所有spike都在有效范围内")

    qm_params = sqm.get_default_qm_params()
    analyzer_mountainsort.compute("quality_metrics", qm_params, n_jobs = 30)

    sexp.export_to_phy(analyzer_mountainsort, output_folder + "/phy_folder_for_kilosort", verbose=True, n_jobs = 30)

Use cache_folder=/tmp/spikeinterface_cache/tmpeagmxpoq/Q2U0ZCD7
write_binary_recording 
engine=process - n_jobs=30 - samples_per_chunk=10,000 - chunk_memory=625.00 KiB - total_memory=18.31 MiB - chunk_duration=1.00s


write_binary_recording (workers: 30 processes): 100%|██████████| 3726/3726 [03:20<00:00, 18.55it/s]


Mountainsort4 use the OLD spikeextractors mapped with NewToOldRecording


estimate_sparsity (no parallelization): 100%|██████████| 3726/3726 [00:00<00:00, 61021.04it/s]
/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/core/basesorting.py:380: UserWarning: The registered recording will not be persistent on disk, but only available in memory
  warnings.warn("The registered recording will not be persistent on disk, but only available in memory")
noise_level (workers: 20 processes): 100%|██████████| 20/20 [00:00<00:00, 515.96it/s]
/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/core/sortinganalyzer.py:2701: DeprecationWarning: The method 'cosine_similarity' is deprecated and will be removed in the next version. Use 'cosine' instead.
  params = self._set_params(**params)
Compute : spike_locations (workers: 30 processes): 100%|██████████| 3726/3726 [00:00<00:00, 9837.78it/s]
/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/qualitymetrics/quality_metric_calculator

所有spike都在有效范围内


...

Mountainsort4 use the OLD spikeextractors mapped with NewToOldRecording


estimate_sparsity (no parallelization): 100%|██████████| 3726/3726 [00:00<00:00, 109833.41it/s]
/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/core/basesorting.py:380: UserWarning: The registered recording will not be persistent on disk, but only available in memory
  warnings.warn("The registered recording will not be persistent on disk, but only available in memory")
noise_level (workers: 20 processes): 100%|██████████| 20/20 [00:00<00:00, 151.11it/s]
/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/core/sortinganalyzer.py:2701: DeprecationWarning: The method 'cosine_similarity' is deprecated and will be removed in the next version. Use 'cosine' instead.
  params = self._set_params(**params)
Compute : spike_locations (workers: 30 processes): 100%|██████████| 3726/3726 [00:00<00:00, 11294.73it/s]
/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/qualitymetrics/quality_metric_calculat

所有spike都在有效范围内


extract PCs (workers: 30 processes): 100%|██████████| 3726/3726 [00:01<00:00, 1917.92it/s]


Run:
phy template-gui  /media/ubuntu/sda/visual_generation/results/neuroscroll_260122/mountainsort/clique_9/phy_folder_for_kilosort/params.py
